# EA2 - Actividad 2.1: Ingesta de Datos

## Objetivos
- Dominar la lectura de multiples formatos de datos (CSV, JSON, Parquet, Excel)
- Comprender la diferencia entre schema inferido y schema manual
- Manejar opciones avanzadas de lectura (delimitador, encoding, nulls)
- Conocer los modos de manejo de datos corruptos
- Escribir y leer datos en formato Parquet

## Conceptos Clave

### Formatos de Datos en Big Data

| Formato | Tipo | Ventajas | Desventajas |
|---------|------|----------|-------------|
| **CSV** | Texto | Universal, legible, simple | Lento, sin tipos, sin compresion |
| **JSON** | Texto | Flexible, soporta anidamiento | Verboso, lento para grandes volumenes |
| **Parquet** | Binario/Columnar | Rapido, comprimido, tipado | No legible directamente |
| **ORC** | Binario/Columnar | Similar a Parquet, optimizado para Hive | Menos adoptado fuera de Hive |
| **Excel** | Binario | Familiar para usuarios de negocio | No escalable, requiere pandas |

### Schema Inferido vs Schema Manual

- **Schema inferido (`inferSchema=True`):** Spark lee una muestra de los datos para adivinar los tipos. Es comodo pero puede ser lento en archivos grandes y cometer errores (ej: confundir strings con integers).
- **Schema manual (`StructType`):** Tu defines explicitamente cada columna y su tipo. Es mas rapido y confiable para produccion.

### Modos de Manejo de Datos Corruptos

| Modo | Comportamiento |
|------|---------------|
| `PERMISSIVE` | Coloca valores null donde hay datos corruptos (default) |
| `DROPMALFORMED` | Elimina filas con datos corruptos |
| `FAILFAST` | Lanza una excepcion al encontrar datos corruptos |

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("EA2_01_ingesta_datos") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

## 1. Leer CSV con Schema Inferido

La forma mas simple de leer un CSV. Spark analiza los datos para inferir automaticamente los tipos de cada columna.

In [ ]:
# Lectura basica de CSV con inferencia de schema
df_flights = spark.read.csv("/home/jovyan/datos/flights.csv", header=True, inferSchema=True)

# Verificar el schema inferido
print("Schema inferido automaticamente:")
df_flights.printSchema()

# Ver las primeras filas
df_flights.show(5)

# Informacion basica
print(f"Filas: {df_flights.count()}")
print(f"Columnas: {len(df_flights.columns)}")

## 2. Leer CSV con Schema Manual

Definir el schema manualmente es una **buena practica en produccion** porque:
1. Es mas rapido (Spark no necesita analizar los datos)
2. Garantiza los tipos correctos
3. Permite controlar la nulabilidad

Se usa `StructType` con una lista de `StructField(nombre, tipo, nullable)`.

In [ ]:
# Definir schema manualmente
schema_flights = StructType([
    StructField("YEAR", IntegerType(), True),
    StructField("MONTH", IntegerType(), True),
    StructField("DAY", IntegerType(), True),
    StructField("DAY_OF_WEEK", IntegerType(), True),
    StructField("AIRLINE", StringType(), True),
    StructField("ORIGIN_AIRPORT", StringType(), True),
    StructField("DESTINATION_AIRPORT", StringType(), True),
    StructField("DEPARTURE_DELAY", DoubleType(), True),
    StructField("ARRIVAL_DELAY", DoubleType(), True),
    StructField("DISTANCE", IntegerType(), True),
])

# Leer con schema manual
df_flights_manual = spark.read.csv("/home/jovyan/datos/flights.csv", header=True, schema=schema_flights)

# Verificar que el schema es exactamente el que definimos
print("Schema manual:")
df_flights_manual.printSchema()

df_flights_manual.show(5)

## 3. Opciones Avanzadas de Lectura

`spark.read.csv()` acepta multiples opciones para manejar archivos con formatos especiales.

In [ ]:
# Ejemplo de opciones avanzadas de lectura CSV
# (Usando el mismo archivo como ejemplo de sintaxis)

df_opciones = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ",") \
    .option("encoding", "UTF-8") \
    .option("nullValue", "NA") \
    .option("dateFormat", "yyyy-MM-dd") \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv("/home/jovyan/datos/flights.csv")

print("Lectura con opciones avanzadas exitosa")
df_opciones.show(5)

**Opciones mas comunes:**

| Opcion | Descripcion | Ejemplo |
|--------|-------------|---------|
| `delimiter` | Separador de campos | `","`, `";"`, `"\t"` |
| `encoding` | Codificacion del archivo | `"UTF-8"`, `"ISO-8859-1"` |
| `nullValue` | String que representa null | `"NA"`, `"NULL"`, `""` |
| `dateFormat` | Formato de fechas | `"yyyy-MM-dd"`, `"dd/MM/yyyy"` |
| `quote` | Caracter de comillas | `'"'` |
| `escape` | Caracter de escape | `'\\'` |
| `multiLine` | Campos con saltos de linea | `"true"` |

## 4. Leer Archivos Excel

Spark no lee Excel nativamente. La estrategia es:
1. Leer con `pandas` (que si soporta Excel)
2. Convertir el DataFrame de pandas a un DataFrame de Spark

In [ ]:
# Leer Excel usando pandas como puente
import pandas as pd

df_pandas = pd.read_excel("/home/jovyan/datos/ufo_sightings.xlsx")
print(f"DataFrame pandas: {df_pandas.shape[0]} filas x {df_pandas.shape[1]} columnas")

# Convertir a DataFrame de Spark
df_ufo = spark.createDataFrame(df_pandas)
print("\nSchema del DataFrame de Spark:")
df_ufo.printSchema()
df_ufo.show(5, truncate=50)

## 5. Escribir y Leer Parquet

**Parquet** es el formato preferido en Big Data porque:
- Es **columnar**: permite leer solo las columnas necesarias
- Tiene **compresion** integrada (snappy, gzip, etc.)
- Preserva los **tipos de datos** en el schema
- Es mucho mas **rapido** de leer que CSV

In [ ]:
# Escribir DataFrame a formato Parquet
df_flights_manual.write.mode("overwrite").parquet("/home/jovyan/datos/flights_parquet")

print("Archivo Parquet escrito exitosamente")

In [ ]:
# Leer desde Parquet
df_parquet = spark.read.parquet("/home/jovyan/datos/flights_parquet")

# Notar que el schema se preserva automaticamente
print("Schema leido desde Parquet (sin necesidad de inferir):")
df_parquet.printSchema()

df_parquet.show(5)
print(f"Filas: {df_parquet.count()}")

## 6. Modos de Manejo de Datos Corruptos

Cuando los datos no coinciden con el schema esperado, Spark ofrece tres estrategias.

In [ ]:
# Modo PERMISSIVE (default): pone null en columnas corruptas
df_permissive = spark.read \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .schema(schema_flights) \
    .csv("/home/jovyan/datos/flights.csv")

print("Modo PERMISSIVE: filas leidas =", df_permissive.count())

In [ ]:
# Modo DROPMALFORMED: elimina filas corruptas
df_drop = spark.read \
    .option("header", "true") \
    .option("mode", "DROPMALFORMED") \
    .schema(schema_flights) \
    .csv("/home/jovyan/datos/flights.csv")

print("Modo DROPMALFORMED: filas leidas =", df_drop.count())

In [ ]:
# Modo FAILFAST: lanza excepcion si encuentra datos corruptos
# Descomentar para probar (puede lanzar error):
# df_fail = spark.read \
#     .option("header", "true") \
#     .option("mode", "FAILFAST") \
#     .schema(schema_flights) \
#     .csv("/home/jovyan/datos/flights.csv")

print("Modo FAILFAST: lanza excepcion al encontrar datos corruptos")

---
## Ejercicios

Ahora es tu turno de practicar. Completa los siguientes ejercicios.

In [ ]:
# =============================================================
# EJERCICIO 1: Leer netflix_titles.csv con schema manual
# =============================================================
# TODO: Define un schema manual usando StructType para el archivo
#   netflix_titles.csv con las siguientes columnas:
#   - show_id (StringType)
#   - type (StringType)
#   - title (StringType)
#   - director (StringType)
#   - cast (StringType)
#   - country (StringType)
#   - date_added (StringType)
#   - release_year (IntegerType)
#   - rating (StringType)
#   - duration (StringType)
#   - listed_in (StringType)
#   - description (StringType)
#
# Luego lee el archivo con ese schema y muestra:
#   1. El schema con printSchema()
#   2. Las primeras 5 filas con show(5, truncate=50)
#   3. El conteo total de filas
#
# Pista: schema_netflix = StructType([StructField("col", Tipo(), True), ...])
#        df = spark.read.csv(path, header=True, schema=schema_netflix)

# Escribe tu codigo aqui:


In [ ]:
# =============================================================
# EJERCICIO 2: Explorar schema inferido de cartas_magic.csv
# =============================================================
# TODO: Lee el archivo cartas_magic.csv con inferSchema=True
#   1. Muestra el schema inferido con printSchema()
#   2. Muestra las primeras 10 filas
#   3. Identifica al menos 2 columnas donde el tipo inferido
#      podria ser incorrecto (escribe tu analisis en un comentario)
#
# Pista: Columnas numericas que Spark lee como string,
#        o columnas de fecha que se leen como string

# Escribe tu codigo aqui:


In [ ]:
# =============================================================
# EJERCICIO 3: Convertir CSV a Parquet y comparar tamanos
# =============================================================
# TODO: 
#   1. Lee flights.csv (con schema manual o inferido)
#   2. Escribe el DataFrame en formato Parquet en:
#      "/home/jovyan/datos/flights_ejercicio_parquet"
#   3. Compara el tamano del archivo CSV original vs los archivos Parquet
#      Usa os.path.getsize() para el CSV y suma los tamanos de los
#      archivos .parquet en la carpeta de salida
#   4. Calcula el porcentaje de compresion
#
# Pista: 
#   import os
#   tamano_csv = os.path.getsize("/home/jovyan/datos/flights.csv")
#   Para Parquet, recorre los archivos en la carpeta con os.listdir()

# Escribe tu codigo aqui:


---
## Desafio Extra (Opcional)

**Para estudiantes avanzados:**

Escribe una funcion `resumen_csv(path)` que reciba la ruta de cualquier archivo CSV y muestre un resumen completo del archivo.

In [ ]:
# =============================================================
# DESAFIO: Funcion de resumen automatico de CSV
# =============================================================
# TODO: Escribe una funcion que reciba un path a un CSV y muestre:
#   1. Numero total de filas
#   2. Numero de columnas
#   3. Nombre y tipo de cada columna
#   4. Cantidad de valores null por columna
#   5. Cantidad de valores unicos por columna (opcional)
#
# Firma: def resumen_csv(spark, path):
#
# Prueba tu funcion con al menos 2 archivos diferentes:
#   - "/home/jovyan/datos/flights.csv"
#   - "/home/jovyan/datos/netflix_titles.csv"
#
# Pista para contar nulls:
#   df.select([F.count(F.when(F.isnull(c), c)).alias(c) for c in df.columns])

# Escribe tu codigo aqui:


---
## Resumen

En esta actividad aprendimos:

1. **Lectura de CSV:** `spark.read.csv(path, header=True, inferSchema=True)` para lectura rapida
2. **Schema manual:** `StructType` + `StructField` para definir tipos explicitamente (recomendado en produccion)
3. **Opciones avanzadas:** delimiter, encoding, nullValue, dateFormat para archivos con formatos especiales
4. **Lectura de Excel:** Usar pandas como puente: `spark.createDataFrame(pd.read_excel(path))`
5. **Parquet:** Formato columnar comprimido, ideal para Big Data: `.write.parquet()` y `.read.parquet()`
6. **Datos corruptos:** Tres modos de manejo: PERMISSIVE (default), DROPMALFORMED, FAILFAST

In [ ]:
# Detener la SparkSession al finalizar
spark.stop()
print("SparkSession detenida correctamente.")